# Meter API — Rate Limiting Demo

This notebook demonstrates Meter API rate limiting: how to read rate-limit headers, how to trigger HTTP 429 errors intentionally, and how to handle them correctly.

## Rate Limit Specification

- **Limit:** 500 requests per minute per API key
- **Timeout:** 60-second request timeout

## Rate-Limit Response Headers

Every API response includes these headers regardless of success or failure:

| Header | Description | Example |
|---|---|---|
| `X-RateLimit-Remaining` | Requests left in the current 60-second window | `"472"` |
| `X-RateLimit-Reset` | RFC 1123 timestamp when the window resets | `"Fri, 07 Mar 2026 12:01:00 GMT"` |
| `Retry-After` | *(HTTP 429 only)* Earliest safe retry time | `"Fri, 07 Mar 2026 12:01:00 GMT"` |

## Correct Handling of HTTP 429

1. Receive HTTP 429 response
2. Parse the `Retry-After` header (RFC 1123 format)
3. Sleep until the `Retry-After` time (or apply exponential back-off if the header is absent)
4. Retry the request **once**
5. **Do NOT** retry in a tight loop — repeated 429s do **not** reset the rate-limit window

## Proactive Approach (Recommended)

Monitor `X-RateLimit-Remaining` on every response. When the value drops below a threshold (e.g. 50), proactively wait for the window reset instead of waiting to hit 429.

## asyncio Design

This notebook uses `asyncio` + `asyncio.to_thread()` (Python 3.9+) to run the synchronous `requests` library on background threads. This enables genuine parallelism for the flood section without blocking the event loop.

## Setup — Imports and Configuration

> ⚠️ **Warning:** The flood section (Section 2) fires 600 concurrent requests to intentionally exhaust your rate-limit window. This will temporarily prevent other requests from succeeding for up to 60 seconds.

In [ ]:
import asyncio
import json
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

import requests
import config

API_URL   = config.API_URL
API_TOKEN = config.API_TOKEN

# Number of concurrent requests to fire in the flood section.
# 500 is the per-minute limit; 600 ensures we exceed the window.
FLOOD_COUNT = 600

# Proactive back-off threshold
PROACTIVE_THRESHOLD = 50

# Request headers used by all requests
HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_TOKEN}",
}

# Simple query used for all requests in this demo
SIMPLE_QUERY = '{ companyBySlug(slug: "meter") { name } }'

print(f"Endpoint  : {API_URL}")
print(f"Rate limit: 500 requests / 60 seconds per API key")
print(f"Flood size: {FLOOD_COUNT} concurrent requests")
print(f"Threshold : slow down below {PROACTIVE_THRESHOLD} remaining")

## Shared Rate-Limit State

These global variables are updated from every response's headers so the proactive back-off logic can check the current remaining count without an extra API call.

The `asyncio.Lock` prevents concurrent coroutines from reading a partially-updated state.

In [ ]:
# Global rate-limit state (updated from response headers)
_rl_remaining = None  # int | None
_rl_reset     = None  # datetime | None
_rl_lock      = asyncio.Lock()  # protects concurrent updates

print("Shared rate-limit state initialised.")

## Header Parsing Helpers

The Meter API uses **RFC 1123** format for `X-RateLimit-Reset` and `Retry-After` timestamps:

```
Fri, 07 Mar 2026 12:01:00 GMT
```

Python's `email.utils.parsedate_to_datetime` handles this format and returns a timezone-aware datetime.

In [ ]:
def parse_rfc1123(value) -> datetime | None:
    """
    Parse an RFC 1123 date string into a timezone-aware datetime.

    Args:
        value: RFC 1123 date string, or None.

    Returns:
        timezone-aware datetime in UTC, or None if unparseable.
    """
    if not value:
        return None
    try:
        return parsedate_to_datetime(value)
    except Exception:
        return None


def seconds_until(dt) -> float:
    """
    Return seconds from now until the given datetime.

    Returns:
        Non-negative float. Returns 0.0 if dt is in the past or None.
    """
    if dt is None:
        return 0.0
    now = datetime.now(timezone.utc)
    return max(0.0, (dt - now).total_seconds())


async def update_rate_limit_state(headers: dict) -> None:
    """
    Update the shared rate-limit counters from a response's headers.
    Called after every request to keep proactive back-off logic informed.
    """
    global _rl_remaining, _rl_reset
    remaining_str = headers.get("X-RateLimit-Remaining")
    reset_str     = headers.get("X-RateLimit-Reset")

    async with _rl_lock:
        if remaining_str is not None:
            try:
                _rl_remaining = int(remaining_str)
            except ValueError:
                pass
        if reset_str:
            _rl_reset = parse_rfc1123(reset_str)


def format_headers(headers: dict) -> str:
    """Format rate-limit headers as a concise one-liner."""
    remaining   = headers.get("X-RateLimit-Remaining", "—")
    reset_str   = headers.get("X-RateLimit-Reset", "—")
    retry_after = headers.get("Retry-After")
    parts = [f"remaining={remaining}", f"reset={reset_str}"]
    if retry_after:
        parts.append(f"retry-after={retry_after}")
    return "  ".join(parts)


# Quick demo
sample = "Fri, 07 Mar 2026 12:01:00 GMT"
dt = parse_rfc1123(sample)
print(f"Parsed '{sample}'")
print(f"  → {dt}")
print(f"  → seconds_until: {seconds_until(dt):.1f}s")

## Core Async Request Function

`single_async_request` executes one GraphQL POST on a background thread via `asyncio.to_thread()`.

**Why `asyncio.to_thread()`?**  
The `requests` library is synchronous (blocking). `asyncio.to_thread()` runs it on the default `ThreadPoolExecutor` without blocking the event loop, allowing many requests to run concurrently within a single async application.

**Why a semaphore?**  
The semaphore limits simultaneous in-flight requests to avoid exhausting OS file descriptors, while still sending requests fast enough to trigger 429 responses.

In [ ]:
async def single_async_request(request_id: int, semaphore: asyncio.Semaphore) -> dict:
    """
    Execute one GraphQL POST request on a background thread.

    Uses asyncio.to_thread() to run requests.post() without blocking
    the event loop. Rate-limit state is updated from response headers.

    Returns:
        Dict with: id, status, remaining, reset, retry_after, error.
    """
    async with semaphore:
        try:
            response = await asyncio.to_thread(
                requests.post,
                API_URL,
                headers=HEADERS,
                json={"query": SIMPLE_QUERY},
                timeout=60,
            )
            status  = response.status_code
            hdrs    = response.headers
            await update_rate_limit_state(hdrs)

            result = {
                "id":          request_id,
                "status":      status,
                "remaining":   hdrs.get("X-RateLimit-Remaining", "—"),
                "reset":       hdrs.get("X-RateLimit-Reset", "—"),
                "retry_after": hdrs.get("Retry-After"),
                "error":       None,
            }

            if status == 200:
                label = "200 OK"
            elif status == 429:
                retry = hdrs.get("Retry-After", "—")
                label = f"429 Too Many Requests  Retry-After: {retry}"
                result["error"] = f"rate limited — retry after {retry}"
            else:
                label = f"HTTP {status}"
                result["error"] = f"unexpected status {status}"

            print(f"  [{request_id:>3}] {label:<55} remaining={result['remaining']}")
            return result

        except requests.Timeout:
            print(f"  [{request_id:>3}] TIMEOUT")
            return {"id": request_id, "status": None, "remaining": None,
                    "reset": None, "retry_after": None, "error": "timeout"}
        except requests.ConnectionError as exc:
            print(f"  [{request_id:>3}] CONNECTION ERROR: {exc}")
            return {"id": request_id, "status": None, "remaining": None,
                    "reset": None, "retry_after": None, "error": str(exc)}


print("single_async_request defined.")

## Retry-After Back-off Function

`request_with_retry` implements the recommended production retry pattern:

1. **Check `X-RateLimit-Remaining` before sending** (proactive back-off). If almost exhausted, wait for the reset timestamp.
2. **On HTTP 429:** read `Retry-After` and sleep until that time.
3. Use exponential back-off as a fallback if `Retry-After` is absent.
4. **Never retry in a tight loop** — the Meter documentation states: *"Repeated 429s will not reset the rate-limit window."*

In [ ]:
async def request_with_retry(max_attempts: int = 5) -> dict | None:
    """
    Execute a query with Retry-After-aware back-off on 429 errors.

    Args:
        max_attempts: Maximum send attempts before giving up.

    Returns:
        Parsed response dict on success, or None after all attempts fail.
    """
    base_delay = 1.0

    for attempt in range(1, max_attempts + 1):
        # ── Proactive check ───────────────────────────────────────────────
        async with _rl_lock:
            remaining = _rl_remaining
            reset_dt  = _rl_reset

        if remaining is not None and remaining < PROACTIVE_THRESHOLD:
            wait_secs = seconds_until(reset_dt) or base_delay
            print(
                f"  ⚠ [Attempt {attempt}] Proactive back-off: only {remaining} requests "
                f"remaining. Waiting {wait_secs:.1f}s for window reset."
            )
            await asyncio.sleep(wait_secs)

        # ── Send the request ──────────────────────────────────────────────
        print(f"  → [Attempt {attempt}/{max_attempts}] Sending request...")

        try:
            response = await asyncio.to_thread(
                requests.post,
                API_URL,
                headers=HEADERS,
                json={"query": SIMPLE_QUERY},
                timeout=60,
            )
        except (requests.Timeout, requests.ConnectionError) as exc:
            print(f"  ✗ Network error on attempt {attempt}: {exc}")
            break

        hdrs   = response.headers
        status = response.status_code
        await update_rate_limit_state(hdrs)

        print(f"  → Response headers: {format_headers(hdrs)}")

        if status == 200:
            body = response.json()
            if "errors" not in body:
                print(f"  ✓ Success on attempt {attempt}.")
                return body

        if status == 429:
            retry_after_dt = parse_rfc1123(hdrs.get("Retry-After"))
            wait_secs = seconds_until(retry_after_dt)
            if wait_secs == 0:
                # Retry-After absent or already past — use exponential back-off
                wait_secs = base_delay * (2 ** (attempt - 1))
                print(
                    f"  ⚠ [Attempt {attempt}] HTTP 429. No valid Retry-After; "
                    f"using exponential back-off: {wait_secs:.1f}s."
                )
            else:
                print(
                    f"  ⚠ [Attempt {attempt}] HTTP 429. "
                    f"Retry-After: {hdrs.get('Retry-After')}. "
                    f"Sleeping {wait_secs:.1f}s."
                )
            await asyncio.sleep(wait_secs)
            continue

        print(f"  ✗ Unexpected HTTP {status} on attempt {attempt}. Stopping.")
        break

    print(f"  ✗ All {max_attempts} attempts failed.")
    return None


print("request_with_retry defined.")

## Section 1 — Observing Rate-Limit Headers

Makes one normal request and displays its rate-limit headers. Establishes a baseline so you can see what the headers look like before any rate-limit pressure is applied.

In [ ]:
async def section_observe_headers() -> None:
    print("Section 1 — Observing Rate-Limit Headers")
    print("Single normal request to read X-RateLimit-Remaining and X-RateLimit-Reset.")
    print()

    semaphore = asyncio.Semaphore(1)
    result    = await single_async_request(1, semaphore)

    print()
    print("Headers on every Meter API response:")
    print(f"  {'X-RateLimit-Remaining':<30} Requests left in the current 60s window")
    print(f"  {'X-RateLimit-Reset':<30} RFC 1123 timestamp when the window resets")
    print(f"  {'Retry-After':<30} RFC 1123 retry time (only on HTTP 429)")
    print()
    print(f"  Current remaining : {result['remaining']}")
    print(f"  Current reset time: {result['reset']}")


await section_observe_headers()

## Section 2 — Triggering HTTP 429 (Flood)

Fires `FLOOD_COUNT` (600) concurrent requests to exhaust the rate limit.

**How it works:**
- `asyncio.gather()` schedules all coroutines concurrently
- Each coroutine calls `asyncio.to_thread()` to run `requests.post()` on a background thread
- A semaphore limits simultaneous in-flight requests to 50 to avoid exhausting OS file descriptors
- Since 600 > 500 (the rate limit), approximately 50 requests should receive HTTP 429

> ⚠️ **This will exhaust your rate-limit window for up to 60 seconds.**

In [ ]:
async def section_flood_to_trigger_429() -> dict:
    print(f"Section 2 — Triggering HTTP 429 ({FLOOD_COUNT} concurrent requests)")
    print(f"The 500 req/min limit means ~{FLOOD_COUNT - 500} requests should receive 429.")
    print(f"Firing {FLOOD_COUNT} requests concurrently (semaphore limits to 50 in-flight)...")
    print()

    semaphore = asyncio.Semaphore(50)
    t_start   = time.monotonic()

    results = await asyncio.gather(
        *[single_async_request(i, semaphore) for i in range(1, FLOOD_COUNT + 1)]
    )

    elapsed = time.monotonic() - t_start

    ok_count    = sum(1 for r in results if r["status"] == 200)
    count_429   = sum(1 for r in results if r["status"] == 429)
    error_count = sum(1 for r in results if r["status"] not in (200, 429) or r["error"])

    print(f"\nFlood results after {elapsed:.1f}s:")
    print(f"  HTTP 200 OK          : {ok_count}")
    print(f"  HTTP 429 rate limited: {count_429}")
    if error_count:
        print(f"  Other / network errors: {error_count}")

    return {"ok": ok_count, "rate_limited": count_429, "errors": error_count}


flood_stats = await section_flood_to_trigger_429()

if flood_stats["rate_limited"] > 0:
    print(f"\n✓ Successfully triggered {flood_stats['rate_limited']} HTTP 429 responses.")
else:
    print("\n⚠ No 429s triggered — the window may have reset between runs.")

## Section 3 — Retry with Retry-After Back-off

Demonstrates the correct retry pattern after triggering 429s.

Steps:
1. Check `X-RateLimit-Remaining` — sleep if below threshold (proactive)
2. Send request
3. On 429: parse `Retry-After`, sleep, retry once
4. Never retry in a tight loop

In [ ]:
print("Section 3 — Retry with Retry-After Back-off")
print("Correct production retry pattern after the flood.")
print()

result = await request_with_retry(max_attempts=5)

if result:
    company = result.get("data", {}).get("companyBySlug", {})
    print(f"\n✓ Final successful response: {company}")
else:
    print("\n⚠ Could not complete after 5 attempts — window may still be exhausted.")
    print("  Wait for X-RateLimit-Reset before retrying in production.")

## Section 4 — Proactive Monitoring Demo

Makes 5 sequential requests while logging the remaining count after each, showing how to monitor the window in a real application.

When `X-RateLimit-Remaining` drops below `PROACTIVE_THRESHOLD` (50), the application should slow down rather than waiting for a 429.

In [ ]:
print("Section 4 — Proactive Monitoring Demo")
print("5 sequential requests logging X-RateLimit-Remaining after each.")
print()

semaphore = asyncio.Semaphore(1)
for i in range(1, 6):
    result = await single_async_request(i, semaphore)
    async with _rl_lock:
        remaining = _rl_remaining
    if remaining is not None and remaining < PROACTIVE_THRESHOLD:
        print(f"  ⚠ Threshold hit ({remaining} < {PROACTIVE_THRESHOLD}). "
              "Would slow down in a real application.")
    await asyncio.sleep(0.2)  # small delay between sequential requests

print()
print("Best practices:")
print("  ✓  Log X-RateLimit-Remaining on every response")
print(f"  ✓  Slow down proactively when remaining < {PROACTIVE_THRESHOLD}")
print("  ✓  On 429: read Retry-After, sleep, retry exactly once")
print("  ✓  Bundle multiple queries into one GraphQL request to reduce count")
print("  ✓  Use asyncio.to_thread() to keep async code non-blocking")
print("  ✗  Do NOT retry immediately in a tight loop after 429")
print("  ✗  Do NOT ignore 429 — repeated 429s do not reset the window")
print("  ✗  Do NOT fire 500+ unchecked requests per minute per key")

## Summary

### Rate Limit Quick Reference

| Item | Value |
|---|---|
| Limit | 500 requests / 60 seconds per API key |
| Header: remaining | `X-RateLimit-Remaining` (integer) |
| Header: reset time | `X-RateLimit-Reset` (RFC 1123 string) |
| Header: retry time | `Retry-After` (RFC 1123 string, 429 only) |
| Proactive threshold | Slow down when remaining < 50 |

### asyncio Pattern

```python
# Run blocking requests.post() without blocking the event loop
response = await asyncio.to_thread(
    requests.post, url, headers=headers, json=payload, timeout=60
)

# Run many requests concurrently
results = await asyncio.gather(
    *[single_request(i, semaphore) for i in range(N)]
)
```

### RFC 1123 Parsing

```python
from email.utils import parsedate_to_datetime
dt = parsedate_to_datetime("Fri, 07 Mar 2026 12:01:00 GMT")
wait_secs = max(0.0, (dt - datetime.now(timezone.utc)).total_seconds())
await asyncio.sleep(wait_secs)
```